In [1]:
import kagglehub, os, torch
dataset_path = kagglehub.dataset_download("sunilthite/ovarian-cancer-classification-dataset")
print("Path to dataset files:", dataset_path)

for dirname, _, filenames in os.walk(dataset_path):
    for filename in filenames[:1]:
        print(os.path.join(dirname, filename))
        

train_dir = os.path.join(dataset_path, "Train_Images")
test_dir = os.path.join(dataset_path, "Test_Images")

/home/conite/anaconda3/envs/GPU_ENV/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/mobilenet_model_224x224_30_10.h5
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Train_Images/LGSC/18690.png
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Train_Images/HGSC/3729.png
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Train_Images/EC/23912.png
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Train_Images/CC/4697.png
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Train_Images/MC/7491.png
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Test_Images/LGSC/1697

In [2]:
import sys
import os
sys.path.append(os.path.abspath('../'))

from src.models.single_network import SingleNetwork
from src.training.trainer import ModelTrainer
from src.processing.preprocess import get_dataloaders, get_transforms
from src.processing.loading import load_image_folder

In [3]:
train_transforms, test_transforms = get_transforms(image_size=224)
data_set_train, idx_to_class_train, class_to_idx_train = load_image_folder(train_dir, train_transforms)
data_set_test, idx_to_class_test, class_to_idx_test = load_image_folder(test_dir, test_transforms)

train_loader, val_loader, cal_loader, test_loader = get_dataloaders(
    dataset={'train': data_set_train, 'test': data_set_test},
    batch_size=64,
    val_split=0.03,
    cal_split=0.12,
    test_split=0.1,
)

Train size: 26523, Val size: 936, Cal size: 3744, Test size: 3082


In [5]:
single_net = SingleNetwork(
    num_classes=len(class_to_idx_train),
    learning_rate=5e-4,
    patience=5,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    train_loader=train_loader,
    val_loader=val_loader,
    cal_loader=cal_loader,
    test_loader=test_loader,
    
)
# single_net.train(epochs=50, model_path='single_net_resnet_sipakmed.pth')
single_net.load('/home/conite/Documents/STAGE/HybridUncertaintyDLFramework/experiments/results/ovarian/best_model_ovarian_resnet_single.pth')

In [6]:
single_net.evaluate()

/home/conite/anaconda3/envs/GPU_ENV/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'accuracy': 0.2426995457495133,
 'f1_score': np.float64(0.10374712978947243),
 'precision': np.float64(0.36027628584152194),
 'recall': np.float64(0.2426995457495133),
 'roc_auc': np.float64(0.4650589511484008),
 'brier_score': np.float32(1.5091103),
 'entropy': array([7.8729680e-15, 1.9173477e-13, 1.6212776e-14, ..., 3.7944897e-14,
        6.9212386e-16, 2.4443452e-15], dtype=float32),
 'uncertainty': array([0.40000004, 0.40000004, 0.40000004, ..., 0.40000004, 0.40000004,
        0.40000004], dtype=float32),
 'ece': np.float64(0.12476979560623071),
 'variance': array([4.79449749e+00, 4.03962517e+01, 1.84516823e+00, 3.62721466e+02,
        1.48169315e+00, 1.20495663e+01, 1.02192078e+02, 4.40677881e+00,
        9.06562195e+02, 3.34955740e+00, 2.80014057e+01, 2.36268555e+02,
        1.27925797e+01, 2.20454932e+03, 7.20238543e+00, 3.71365242e+01,
        3.34128235e+02, 1.57337761e+01, 3.02987231e+03, 1.53777256e+01,
        6.56021347e+01, 6.14897644e+02, 2.98655148e+01, 5.61161816e+03,